In [2]:
import os
from dotenv import load_dotenv
from langsmith import Client
opanai_key = os.getenv("OPENAI_PI_KEY")
langsmith_key = os.getenv("LANGSMITH_API_KEY")
langsmith_tracing = True
load_dotenv(override=True)

True

In [3]:
client = Client()
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id= dataset.id,
    examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }        
    ]
)

{'example_ids': ['f3f43682-54df-4adc-97b8-1fd0db8260a4',
  'd7a2a774-0841-4fff-91f5-ae3b33d74efe',
  'c4352cf3-5642-44ea-bf09-6c2bf4b14ff6',
  '2e5afc60-f9c5-49f9-9ae6-f8d31f3c94c2',
  'c94acdb6-dd13-49bd-b427-a6963f77c948'],
 'count': 5,
 'as_of': '2026-06-08T10:21:49.948558711Z'}

## Define Metrics (LLM as a Judge) ##

In [4]:
import openai
from langsmith import wrappers

openai_client =wrappers.wrap_openai(openai.OpenAI())
eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict)-> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role":"system", "content":eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content
    
    return response == "CORRECT"
    

In [5]:
# Concisions : Checks whether the actual output is less than 2x the length of expected result.

def concision(outputs: dict, reference_outputs: dict)-> bool:
    return int(len(outputs["response"])) < 2 * len(reference_outputs["answer"])

In [10]:
## Running the Evaluation ##
default_instructions = "Respond to the users question in a short, concise manner(one short sentence)."
def my_app(question: str, instructions: str = default_instructions)-> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system", "content": instructions},
            {"role":"user", "content":question}
        ]
    ).choices[0].message.content
    return response

In [13]:
# calling my_app for every datapoints
def ls_target(inputs: str)-> dict:
    return {"response": my_app(inputs["question"])}

In [14]:
## Run for evaluation
experimental_results= client.evaluate(
    ls_target, # AI System
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix= "openai-4o-mini"
)

experimental_results

View the evaluation results for experiment: 'openai-4o-mini-a97e2a4d' at:
https://smith.langchain.com/o/ded64c70-97b2-4828-9487-f10ed0ed0c43/datasets/224c90e8-6402-44c9-afa8-5487c04ffac4/compare?selectedSessions=1f4edbe3-267d-4b59-96de-65b2bdc71ce0




5it [00:12,  2.55s/it]


<ExperimentResults openai-4o-mini-a97e2a4d>

## Evaluation For RAG ##

In [16]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of Urls to load documents from

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/"    
]

#load documents from Urls
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

#Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

#split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

#Add the document chunks to the "vector store" using the OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents = doc_splits,
    embeddings = OpenAIEmbeddings()
)

#with using langchain we can turn vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

ConnectTimeout: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Max retries exceeded with url: /gpt-2/encodings/main/encoder.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='openaipublic.blob.core.windows.net', port=443) at 0x2070956c470>, 'Connection to openaipublic.blob.core.windows.net timed out. (connect timeout=None)'))